# 동결 문장 임베딩 비교

이번 실험은 **입력 표현만** 바꿉니다. 기존 문자 TF-IDF 대신 동결된
`intfloat/multilingual-e5-small`의 384차원 벡터를 쓰고, 라벨·8/1/1 LODO·balanced
가중치·Logistic/LinearSVC·평가 지표는 그대로 둡니다. 인코더 파인튜닝은 하지 않습니다.

모델 파일 약 470MB는 USB에 둡니다. 아래 명령을 먼저 한 번 실행하면, 재생성 가능한
임베딩만 `data/processed/multilingual-e5-small.npz`에 저장됩니다.

```powershell
python -m scripts.evaluation.embeddings --model-cache E:\rfp-models
```

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))

from scripts.evaluation.baselines import CHAR_BALANCED, run_lodo, summarize
from scripts.evaluation.embeddings import (E5_LOGISTIC, E5_SVM, load_cached_embeddings,
                                               run_embedding_lodo)
from scripts.labeling.label_dataset import load_label_dataset

rows, meta = load_label_dataset()
cache = ROOT / 'data/processed/multilingual-e5-small.npz'
if not cache.exists():
    raise FileNotFoundError('README의 embeddings 명령을 먼저 실행하세요')
embeddings = load_cached_embeddings(cache, rows)
if embeddings is None:
    raise ValueError('임베딩 캐시가 현재 데이터·모델 설정과 다릅니다. 다시 생성하세요')

runs = {
    CHAR_BALANCED.name: run_lodo(rows, CHAR_BALANCED),
    E5_LOGISTIC.name: run_embedding_lodo(rows, embeddings, E5_LOGISTIC),
    E5_SVM.name: run_embedding_lodo(rows, embeddings, E5_SVM),
}
table = pd.DataFrame({name: {metric: values['fold_mean']
                                  for metric, values in summarize(run).items()}
                      for name, run in runs.items()}).T
display(table[['macro_f1', 'accuracy', 'review_precision', 'review_recall', 'review_f1']]
        .style.format('{:.3f}'))

In [ ]:
baseline = runs[CHAR_BALANCED.name]
comparisons = []
for name in (E5_LOGISTIC.name, E5_SVM.name):
    deltas = [after.macro_f1 - before.macro_f1
              for before, after in zip(baseline, runs[name])]
    recall_deltas = [after.review_recall - before.review_recall
                     for before, after in zip(baseline, runs[name])]
    comparisons.append({
        '비교': name, 'macroF1 평균차': sum(deltas) / len(deltas),
        '최소': min(deltas), '최대': max(deltas),
        'macroF1 우세': f'{sum(d > 0 for d in deltas)}/10',
        '계약recall 평균차': sum(recall_deltas) / len(recall_deltas),
    })
display(pd.DataFrame(comparisons).set_index('비교').style.format({
    'macroF1 평균차': '{:+.3f}', '최소': '{:+.3f}', '최대': '{:+.3f}',
    '계약recall 평균차': '{:+.3f}',
}))

## 결론

E5 + Logistic은 macro F1 **0.544**, E5 + LinearSVC는 **0.546**으로 문자 TF-IDF +
Logistic의 **0.601**을 넘지 못했습니다. E5 Logistic의 macro F1 차이는 평균 -0.057이고
10개 fold 중 2개에서만 우세했습니다. 계약 recall도 0.514에서 0.443으로 낮아졌습니다.

이 결과는 **고정된 범용 임베딩을 그대로 특징으로 쓰는 방법**이 현재 데이터에서 약하다는
뜻입니다. 인코더 자체를 라벨에 맞춰 학습하는 파인튜닝까지 나쁘다는 뜻은 아닙니다. 현재
주 기준선은 문자 3~4gram + balanced Logistic으로 유지합니다.